# ClusterAlgebras.jl - Feature Overview

A brief tour of every exported symbol.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using ClusterAlgebras, AbstractAlgebra

## 1  Quiver construction

A quiver is encoded by its skew-symmetrizable exchange matrix `B`. The mutable vertices drive mutation; frozen vertices appear in the exchange relations but are never mutated themselves.

In [ ]:
# From a skew-symmetric integer matrix
B = [0 1 0; -1 0 1; 0 -1 0]   # A₃ exchange matrix
q_mat = Quiver(B)

In [ ]:
# From an edge list  (src, dst[, weight])
q_edges = Quiver([(1,2), (2,3)])

In [ ]:
# Named Dynkin quivers  (simply-laced)
q_A3 = Quiver(:A, 3)
q_D4 = Quiver(:D, 4)
q_E6 = Quiver(:E, 6)

# Non-simply-laced  (carries symmetrizer d)
q_B3 = Quiver(:B, 3)
q_G2 = Quiver(:G, 2)

In [ ]:
# String shorthand
q_F4 = Quiver("F4")

In [ ]:
# Frozen vertices: first n_mutable rows/cols are mutable, the rest are frozen
B_ext = [0 1 1; -1 0 0; -1 0 0]   # A₂ with one frozen vertex
q_frozen = Quiver(B_ext, 2)        # vertices 1,2 mutable; vertex 3 frozen

## 2  Quiver accessors

Thin functional API over the struct fields; `to_dot` produces a Graphviz string for visualization.

In [ ]:
q = Quiver(:A, 3)
nvertices(q)        # total number of vertices

In [ ]:
labels(q)           # vertex labels (strings)

In [ ]:
is_frozen(q, 1), is_frozen(q_frozen, 3)

In [ ]:
# Graphviz DOT export
println(to_dot(Quiver(:A, 3)))

## 3  Seeds & mutation

A seed `(Q, x)` pairs a quiver with a cluster of rational functions. Mutation at vertex `k` replaces `xₖ` by the exchange relation `(∏ xⱼ^{[Bⱼₖ]₊} + ∏ xⱼ^{[-Bⱼₖ]₊}) / xₖ`. By the Laurent phenomenon the result always lies in `ZZ[x₁±¹, …, xₙ±¹]`.

In [ ]:
s0 = Seed(Quiver(:A, 2))
s0

In [ ]:
# Custom variable names
s_named = Seed(Quiver(:A, 2), ["a", "b"])
s_named

In [ ]:
# Mutation by index
s1 = mutate(s0, 1)
s1

In [ ]:
# Mutation by label
mutate(s0, "2")

In [ ]:
# A₂ has exactly 5 cluster variables; the mutation sequence of length 6 returns to the initial seed
s_seq = mutate(s0, [1, 2, 1, 2, 1, 2])
s_seq

In [ ]:
# mutation_path is tracked automatically
s1.mutation_path

In [ ]:
# Seeds are iterable collections of cluster variables
collect(s1)         # Vector of rational functions

In [ ]:
s1[1], s1[2]        # index into the cluster

## 4  LaTeX display

Seeds render their cluster variables in aligned LaTeX - useful in notebooks for reading off explicit rational functions.

In [ ]:
# In a Jupyter notebook the MIME"text/latex" method renders automatically.
# Trigger it explicitly:
display("text/latex", repr(MIME("text/latex"), mutate(Seed(Quiver(:A, 3)), [1, 2, 3])))

## 5  ExtendedSeed: c-vectors and g-vectors

`ExtendedSeed` augments a seed with its **C-matrix** (tropical/sign data) and **G-matrix** (grading data), updated combinatorially alongside every mutation. The sign-coherence theorem (Fomin–Zelevinsky) guarantees each c-vector is either entirely ≥ 0 or entirely ≤ 0 - a useful correctness check.

In [ ]:
es0 = extend(Seed(Quiver(:A, 3)))
es1 = mutate(es0, [1, 2, 3])
es1

In [ ]:
cmatrix(es1)        # C-matrix; columns are c-vectors

In [ ]:
gmatrix(es1)        # G-matrix; columns are g-vectors

In [ ]:
cvectors(es1)       # c-vectors as a list

In [ ]:
gvectors(es1)       # g-vectors as a list

In [ ]:
is_sign_coherent(es1)   # should always be true (FZ4 theorem)

In [ ]:
c_vector(es1, 1)    # k-th c-vector (column k of C-matrix)

In [ ]:
g_vector(es1, 2)    # k-th g-vector (column k of G-matrix)

In [ ]:
# Rational y-variables in Frac(ZZ[y₁,…,yₙ])
y_variables(es1)

In [ ]:
# Tropical y-variables = c-vectors (the sign data driving wall-crossing)
y_variables(es1; semifield=:tropical)

## 6  F-polynomials

The **F-polynomial** of a cluster variable is obtained by setting all initial cluster variables to 1 under the *principal-coefficient* specialisation. The result is always a polynomial with non-negative integer coefficients and constant term 1 (positivity theorem).

In [ ]:
# A₂: compute F-polynomials for each of the 5 seeds in the exchange graph
s_A2 = Seed(Quiver(:A, 2))
for path in [Int[], [1], [2], [1,2], [2,1]]
    es = extend(mutate(s_A2, path))
    fps = fpolynomials(es)
    println("path=$path  →  F = $fps")
end

In [ ]:
# Individual F-polynomial by index
es_A2 = extend(mutate(Seed(Quiver(:A, 2)), [1, 2]))
f_polynomial(es_A2, 1), f_polynomial(es_A2, 2)

## 7  Separation formula

The **separation formula** (Fomin–Zelevinsky, *Cluster algebras IV*, Cor. 6.3) expresses every
cluster variable in closed factored form:

$$x_k = \Bigl(\prod_i x_i^{g_i}\Bigr) \cdot F_k(\hat y_1,\dots,\hat y_n)$$

where $\hat y_j = y_j \prod_i x_i^{B_0[i,j]}$ and $g_i$ are the components of the g-vector.
`separation_formula_trivial` specialises $y \to 1$; its output equals the cluster variable
from direct mutation. `separation_formula` keeps $y$ symbolic.

In [ ]:
# Trivial specialisation (y → 1): result equals direct mutation
es_A3 = extend(mutate(Seed(Quiver(:A, 3)), [1, 2, 3]))
s_dir = mutate(Seed(Quiver(:A, 3)), [1, 2, 3])
for k in 1:3
    sf  = separation_formula_trivial(es_A3, k)
    println("k=", k, ": formulas agree? ", string(sf) == string(s_dir.cluster[k]))
end

In [ ]:
# Full formula with y variables kept symbolic (used in wall-crossing / scattering diagrams)
separation_formula(es_A3, 1)

## 8  Denominator vectors

Every cluster variable, written as a reduced fraction of polynomials in the initial cluster, has a denominator of the form `x₁^{d₁}⋯xₙ^{dₙ}`. The exponent vector `(d₁,…,dₙ)` is the **denominator vector**; it uniquely identifies the cluster variable in finite type.

In [ ]:
using ClusterAlgebras: denominator_vector

s = mutate(Seed(Quiver(:A, 3)), [1, 2, 3])
[denominator_vector(s, k) for k in 1:3]

## 9  Root systems

For a finite Dynkin quiver, the non-initial cluster variables are in bijection with the positive roots of the associated root system (Fomin–Zelevinsky's cluster-root correspondence). `RootSystem` computes positive roots via BFS on the Cartan matrix and stores classical Lie-theoretic invariants.

In [ ]:
rs = RootSystem(:A, 3)
rs.positive_roots

In [ ]:
rs.coxeter_number, rs.exponents, rs.weyl_group_order

In [ ]:
almost_positive_roots(rs)   # negative simples ∪ positive roots

In [ ]:
# The Cartan companion of a quiver
cartan_companion(Quiver(:B, 3))

## 10  Finite / affine type detection

A cluster algebra is of **finite type** (finitely many cluster variables) iff the symmetrized Cartan companion `D·A` is positive definite - equivalently, the Cartan type is a finite Dynkin diagram. **Affine type** corresponds to positive semi-definite of corank 1.

In [ ]:
is_finite_type(Quiver(:A, 3))          # true

In [ ]:
# Kronecker quiver: affine type Ã₁
q_kron = Quiver([0 2; -2 0])
is_finite_type(q_kron), is_affine_type(q_kron)

In [ ]:
# Markov quiver: mutation-finite but neither finite nor affine type
q_markov = Quiver([0 2 -2; -2 0 2; 2 -2 0])
is_finite_type(q_markov), is_affine_type(q_markov)

In [ ]:
cartan_type(Quiver(:D, 4))   # (:D, 4)

In [ ]:
cartan_type(Quiver(:G, 2))   # (:G, 2)

## 11  Mutation class & exchange graph

The **mutation class** (quiver-level BFS) groups all quivers reachable by mutation; the **exchange graph** (seed-level BFS) is the graph whose vertices are clusters and whose edges are single mutations. For finite-type algebras both BFS terminate; the cluster counts match known Catalan-type formulas (e.g. 5 for A₂, 14 for A₃, 50 for D₄).

In [ ]:
# Quiver-level BFS
mc = mutation_class(Quiver(:A, 2))
length(mc), is_truncated(mc)    # 2 quivers, not truncated

In [ ]:
# Seed-level BFS (exchange graph)
eg_A2 = exchange_graph(Seed(Quiver(:A, 2)))
length(eg_A2)   # 5 clusters

In [ ]:
eg_D4 = exchange_graph(Seed(Quiver(:D, 4)))
length(eg_D4)   # 50 clusters

In [ ]:
# Truncation example
eg_inf = exchange_graph(Seed(q_markov); max_seeds=50)
is_truncated(eg_inf)

In [ ]:
# Mutation-finiteness test (BFS cutoff at 10 000 quivers)
is_mutation_finite(Quiver(:D, 4))         # true

In [ ]:
is_mutation_finite(Quiver([0 1 1; -1 0 3; -1 -3 0]))   # false (mutation-infinite)

## 12  Enumerative invariants

For finite-type cluster algebras, the number of cluster variables and clusters are given by
exact closed formulas in terms of the Coxeter number $h$ and degrees $d_i = e_i + 1$:

$$\text{\# cluster variables} = \frac{n(h+2)}{2}, \qquad
  \text{\# clusters} = \prod_{i=1}^n \frac{h + d_i}{d_i} \quad (\text{W-Catalan number})$$

The **f-vector** of the cluster complex counts faces by dimension; the **h-vector** entries
are the W-Narayana numbers, which are non-negative and sum to the W-Catalan number.

In [ ]:
# Closed-form counts from the root system
for (t, n) in [(:A,3), (:D,4), (:E,6), (:E,8)]
    rs = RootSystem(t, n)
    println("$t$n:  #vars = ", n_cluster_variables(rs), ",  #clusters = ", n_clusters(rs))
end

In [ ]:
# Or directly from the quiver (runs cartan_type internally)
n_cluster_variables(Quiver(:A, 3)), n_clusters(Quiver(:A, 3))   # 9, 14

In [ ]:
# f-vector and h-vector of the cluster complex from an exchange graph
fv = f_vector(eg_A2)   # [f_{-1}, f_0, f_1] = [1, 5, 5]
hv = h_vector(eg_A2)   # [1, 3, 1]  (Narayana numbers for A₂)
println("A₂ f-vector: ", fv)
println("A₂ h-vector: ", hv, "  (sum = ", sum(hv), ")")

## 13  Conway–Coxeter friezes

A **frieze pattern** is an array of positive integers in which every unit diamond `ad - bc = 1`. Conway and Coxeter showed these are in bijection with triangulations of a polygon; the top row of a frieze (the *quiddity sequence*) encodes the triangulation.

In [ ]:
# Fan triangulation of a hexagon
f6 = frieze(6)
f6

In [ ]:
AbstractAlgebra.is_valid(f6)   # unimodular rule holds everywhere

In [ ]:
# Custom quiddity sequence
f_custom = frieze([3, 2, 1, 2])
f_custom

## 14  Error types

Package-specific exceptions carry structured fields and render informative messages via `showerror`.

In [ ]:
# Attempting to mutate a frozen vertex
try
    mutate(Seed(q_frozen), 3)   # vertex 3 is frozen
catch e
    showerror(stdout, e); println()
end

In [ ]:
# Non-skew-symmetrizable matrix
try
    Quiver([0 2; -3 0])   # 2 ≠ 3
catch e
    showerror(stdout, e); println()
end

In [ ]:
# Unknown vertex label
try
    mutate(Seed(Quiver(:A, 2)), "z")
catch e
    showerror(stdout, e); println()
end